# InsertionSort (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/InsertionSort_cpp.ipynb)

Capitolo 13 — Correttezza. Notebook eseguibile di `InsertionSort`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/InsertionSort_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/InsertionSort_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Indici.** Nello pseudocodice gli array sono indicizzati da 1; nel codice C++ da 0: l'elemento `a[i]` dello pseudocodice è `a[i-1]` nel codice (e un ciclo `while i <= n` da 1 diventa `while i < n` da 0). Le tabelle stampate usano gli indici dello pseudocodice.

## Ordinamento per inserimento: InsertionSort

**Input:** un array $a[1..n]$ di $n \ge 1$ numeri naturali. **Output:** l'array ordinato in modo non decrescente. **Invariante esterno:** $a[1..i-1]$ contiene gli elementi originali di $a[1..i-1]$ ordinati e $a[i..n]$ è uguale all'originale. **Invariante interno** (I1)–(I3): $a[1..j] = b[1..j]$, $a[j+2..i] = b[j+1..i-1]$ e questi elementi sono $> key$.

Pseudocodice: sezione 13.3 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [1]:
%%writefile insertion_sort.cpp
#include <algorithm>
#include <cstdio>
#include <string>
using namespace std;

const int MAXN = 64;

// "[a[from], ..., a[to-1]]"
string Fmt(const int* a, int from, int to)
{
    string s = "[";
    for (int i = from; i < to; i++)
    {
        if (i > from) s += ", ";
        s += to_string(a[i]);
    }
    return s + "]";
}

bool Equal(const int* x, const int* y, int len)     // x[0..len-1] == y[0..len-1]
{
    for (int k = 0; k < len; k++)
        if (x[k] != y[k]) return false;
    return true;
}

// L'array con una barra dopo la parte ordinata a[1..i-1]
string Split(const int* a, int n, int i)
{
    string s = Fmt(a, 0, i);
    s.pop_back();
    string t = Fmt(a, i, n);
    return s + " | " + t.substr(1);
}

// Invariante esterno: a[1..i-1] = elementi originali di a[1..i-1] ordinati, a[i..n] = originale
void PrintOuter(const int* a, const int* orig, int n, int i)
{
    int s[MAXN];
    for (int k = 0; k < i; k++) s[k] = orig[k];
    sort(s, s + i);
    bool ok = Equal(a, s, i) && Equal(a + i, orig + i, n - i);
    printf("%2d | %-26s | %-19s | %s\n", i + 1, Split(a, n, i).c_str(), Fmt(orig, 0, i).c_str(), ok ? "ok" : "FALSA");
}

// Invariante interno (I1)-(I3); b = a a inizio iterazione
void PrintInner(const int* a, const int* b, int n, int i, int j, int key)
{
    int ip = i + 1, jp = j + 1;                    // indici dello pseudocodice
    bool ok = Equal(a, b, jp) && Equal(a + jp + 1, b + jp, ip - jp - 1);
    for (int k = jp + 1; k < ip; k++)
        if (a[k] <= key) ok = false;
    string cond = "j = 0: false";
    if (j >= 0)
    {
        bool t = a[j] > key;
        cond = "a[" + to_string(jp) + "] = " + to_string(a[j]) + (t ? " > " : " <= ") + to_string(key) + ": " + (t ? "true" : "false");
    }
    printf("%d | %-15s | %-12s | %-9s | %-20s | %s\n", jp, Fmt(a, 0, n).c_str(), Fmt(a, 0, jp).c_str(), Fmt(a, jp + 1, ip).c_str(), cond.c_str(), ok ? "ok" : "FALSA");
}

// InsertionSort(a, n)
void InsertionSort(int a[], int n, bool showOuter = true, int traceInnerAt = 0)
{
    int orig[MAXN], b[MAXN];
    for (int k = 0; k < n; k++) orig[k] = a[k];    // copia dell'input: serve solo a verificare l'invariante
    int i = 1;                                     // i := 2
    if (showOuter) PrintOuter(a, orig, n, i);
    while (i < n)                                  // i <= n
    {
        for (int k = 0; k < n; k++) b[k] = a[k];   // contenuto di a a inizio iterazione (solo per l'invariante interno)
        int key = a[i];
        int j = i - 1;
        bool tr = (i + 1 == traceInnerAt);
        if (tr) PrintInner(a, b, n, i, j, key);
        while (j >= 0 && a[j] > key)               // j >= 1 and a[j] > key
        {
            a[j + 1] = a[j];
            j = j - 1;
            if (tr) PrintInner(a, b, n, i, j, key);
        }
        a[j + 1] = key;
        i = i + 1;
        if (showOuter) PrintOuter(a, orig, n, i);
    }
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Il ciclo esterno: a[1..5] = (5, 2, 4, 1, 3), esempio delle dispense
    int a[] = {5, 2, 4, 1, 3};
    int n = 5;
    printf("a = %s\n", Fmt(a, 0, n).c_str());
    printf(" i | a (ordinato | resto)       | a[1..i-1] originale | invariante\n");
    InsertionSort(a, n);
    printf("ordinato: %s\n", Fmt(a, 0, n).c_str());

    // Il ciclo interno all'iterazione i = 5, con key = 3 (a = (1, 2, 4, 5, 3) all'inizio)
    printf("\n");
    int c[] = {5, 2, 4, 1, 3};
    printf("j | a               | a[1..j]      | a[j+2..i] | condizione           | invariante\n");
    InsertionSort(c, n, false, 5);
    printf("dopo la riga 10 (a[j+1] = key): a = %s\n", Fmt(c, 0, n).c_str());
    return 0;
}
#endif

Writing insertion_sort.cpp


In [2]:
!g++ -std=c++17 -O2 -o insertion_sort insertion_sort.cpp && ./insertion_sort

a = [5, 2, 4, 1, 3]
 i | a (ordinato | resto)       | a[1..i-1] originale | invariante
 2 | [5 | 2, 4, 1, 3]           | [5]                 | ok
 3 | [2, 5 | 4, 1, 3]           | [5, 2]              | ok
 4 | [2, 4, 5 | 1, 3]           | [5, 2, 4]           | ok
 5 | [1, 2, 4, 5 | 3]           | [5, 2, 4, 1]        | ok
 6 | [1, 2, 3, 4, 5 | ]         | [5, 2, 4, 1, 3]     | ok
ordinato: [1, 2, 3, 4, 5]

j | a               | a[1..j]      | a[j+2..i] | condizione           | invariante
4 | [1, 2, 4, 5, 3] | [1, 2, 4, 5] | []        | a[4] = 5 > 3: true   | ok
3 | [1, 2, 4, 5, 5] | [1, 2, 4]    | [5]       | a[3] = 4 > 3: true   | ok
2 | [1, 2, 4, 4, 5] | [1, 2]       | [4, 5]    | a[2] = 2 <= 3: false | ok
dopo la riga 10 (a[j+1] = key): a = [1, 2, 3, 4, 5]


**Prova tu.** Cambia l'array: ogni riga della tabella esterna deve restare `ok`.

In [3]:
%%writefile prova_insertion_sort.cpp
#define SOLO_LIBRERIA   // non eseguire il main di insertion_sort.cpp
#include "insertion_sort.cpp"

int main()
{
    int a[] = {6, 3, 8, 1, 9, 2};      // <-- cambia l'array
    int n = 6;
    printf(" i | a (ordinato | resto)       | a[1..i-1] originale | invariante\n");
    InsertionSort(a, n);
    return 0;
}

Writing prova_insertion_sort.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_insertion_sort prova_insertion_sort.cpp && ./prova_insertion_sort

 i | a (ordinato | resto)       | a[1..i-1] originale | invariante
 2 | [6 | 3, 8, 1, 9, 2]        | [6]                 | ok
 3 | [3, 6 | 8, 1, 9, 2]        | [6, 3]              | ok
 4 | [3, 6, 8 | 1, 9, 2]        | [6, 3, 8]           | ok
 5 | [1, 3, 6, 8 | 9, 2]        | [6, 3, 8, 1]        | ok
 6 | [1, 3, 6, 8, 9 | 2]        | [6, 3, 8, 1, 9]     | ok
 7 | [1, 2, 3, 6, 8, 9 | ]      | [6, 3, 8, 1, 9, 2]  | ok


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).